In [1]:
# Import required packages
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = os.path.join(PROJECT_DIR, "Data")
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "merged_master.pkl")

# Parallel processing
N_JOBS = 16  # Number of parallel jobs for OOS predictions

# Ridge parameters
ALPHAS = np.logspace(-6, 6, 100)  # Alpha range for Ridge CV
CV_FOLDS = 5  # Number of cross-validation folds

In [3]:
data = pd.read_pickle(INPUT_DATA)

In [4]:
# Prepare features and target
# Target variable
TARGET = 'f_cumret1'

# Select features
# Exclude non-feature columns and potential other targets
NON_FEATURES = ['date', 'ticker', 'permno', 'shrout', 'prc'] 

# Identify numeric columns
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

# Filter features
FEATURES = []
for c in numeric_cols:
    if c == TARGET:
        continue
    if c in NON_FEATURES:
        continue
    # Exclude other return/abnormal return columns to prevent leakage
    # Assuming targets start with 'ret' or 'ar_'
    if c.startswith('ret') or c.startswith('ar_'):
        continue
    FEATURES.append(c)

# Remove missing values
model_data = data[[TARGET] + FEATURES + ['date', 'ticker']].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Number of features: {len(FEATURES)}")
print(f"Features: {FEATURES}")

Sample size: 16,743,676
Target: f_cumret1
Number of features: 31
Features: ['net_sentiment', 'extreme_bullish_80', 'extreme_bullish_90', 'extreme_bearish_80', 'extreme_bearish_90', 'disagreement_index', 'raw_volume', 'log_volume', 'unique_user_count', 'volume_diff', 'log_volume_change', 'abn_volume_5d', 'abn_attention_std_5d', 'attention_surge_5d', 'abn_volume_21d', 'abn_attention_std_21d', 'attention_surge_21d', 'abn_volume_63d', 'abn_attention_std_63d', 'attention_surge_63d', 'abn_volume_250d', 'abn_attention_std_250d', 'attention_surge_250d', 'silence_gap_hours', 'relative_volume', 'attention_hhi', 'abnormal_sentiment_1d', 'abnormal_sentiment_5d', 'abnormal_sentiment_21d', 'abnormal_sentiment_63d', 'abnormal_sentiment_250d']


In [5]:
# Add date column and sort
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])
model_data = model_data.sort_values('date')
model_data['year_month'] = model_data['date'].dt.to_period('M')

# In-Sample Ridge regression

In [6]:
# In-sample training window (adjustable)
TRAIN_START_DATE = '2011-01-01'
TRAIN_END_DATE = '2011-12-31'

# Filter data to training window
train_start = pd.to_datetime(TRAIN_START_DATE)
train_end = pd.to_datetime(TRAIN_END_DATE)
train_mask = (model_data['date'] >= train_start) & (model_data['date'] <= train_end)

X_train = model_data.loc[train_mask, FEATURES]
y_train = model_data.loc[train_mask, TARGET]

print(f"Training window: {TRAIN_START_DATE} to {TRAIN_END_DATE}")
print(f"Training samples: {len(X_train):,}")
print(f"Number of features: {len(FEATURES)}")

# Normalize features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Fit Ridge regression model with cross-validation for optimal alpha
import time
start_time = time.time()

ridge_model = RidgeCV(alphas=ALPHAS, cv=CV_FOLDS)
ridge_model.fit(X_train_scaled, y_train)

elapsed_time = time.time() - start_time

# Make predictions
y_pred = ridge_model.predict(X_train_scaled)

# Evaluate performance
r2 = r2_score(y_train, y_pred)
mse = mean_squared_error(y_train, y_pred)
rmse = np.sqrt(mse)

print(f"\nIn-Sample Ridge Regression Results")
print("=" * 50)
print(f"Optimal alpha (regularization): {ridge_model.alpha_:.6f}")
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print("\nTop 10 Coefficients (on standardized features):")
coef_df = pd.DataFrame({'feature': FEATURES, 'coef': ridge_model.coef_})
coef_df['abs_coef'] = coef_df['coef'].abs()
print(coef_df.sort_values('abs_coef', ascending=False).head(10)[['feature', 'coef']])
print(f"\nIntercept: {ridge_model.intercept_:.6f}")
print(f"\nTraining time: {elapsed_time:.2f} seconds")

Training window: 2011-01-01 to 2011-12-31
Training samples: 1,083,247
Number of features: 31

In-Sample Ridge Regression Results
Optimal alpha (regularization): 1000000.000000
R-squared: 0.000051
RMSE: 0.038576
MSE: 0.001488

Top 10 Coefficients (on standardized features):
                    feature      coef
15    abn_attention_std_21d  0.000054
0             net_sentiment -0.000046
30  abnormal_sentiment_250d -0.000039
1        extreme_bullish_80 -0.000038
11            abn_volume_5d  0.000038
12     abn_attention_std_5d  0.000037
2        extreme_bullish_90 -0.000037
27    abnormal_sentiment_5d  0.000032
8         unique_user_count -0.000032
29   abnormal_sentiment_63d -0.000030

Intercept: -0.000204

Training time: 86.31 seconds


# OOS predictions

In [7]:
# Out-of-sample predictions with MONTHLY TRAINING but DAILY PREDICTIONS
# The model is trained once per month (at month-end) and used to predict all days in the following month

# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW = 252  # Rolling window size in trading days (252 = one year)

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

# Get unique months in the OOS period
oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Rolling window: {WINDOW} trading days")
print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")
print(f"Number of features: {len(FEATURES)}")

def train_and_predict_month(month_idx, pred_month, model_data, unique_dates, oos_dates,
                            FEATURES, TARGET, WINDOW, ALPHAS, CV_FOLDS):
    """Train Ridge model for a single month and generate predictions for all days in that month."""
    predictions = []
    alpha_info = None
    
    # Get all prediction dates in this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    
    if len(month_dates) == 0:
        return predictions, alpha_info
    
    # Training cutoff: end of the previous month (first day of pred_month - 1 day)
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)
    
    # Find the last trading day before or on train_cutoff
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return predictions, alpha_info
    last_train_date = train_dates[-1]
    
    # ROLLING WINDOW: Train on last WINDOW trading days before the month
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx < WINDOW:
        return predictions, alpha_info
    
    start_date = unique_dates[last_train_date_idx - WINDOW + 1]
    train_mask = (model_data['date'] >= start_date) & (model_data['date'] <= last_train_date)
    X_train = model_data.loc[train_mask, FEATURES]
    y_train = model_data.loc[train_mask, TARGET]
    
    if len(X_train) == 0:
        return predictions, alpha_info
    
    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Fit Ridge with CV
    ridge_model = RidgeCV(alphas=ALPHAS, cv=CV_FOLDS)
    ridge_model.fit(X_train_scaled, y_train)
    alpha_info = {'month': pred_month, 'alpha': ridge_model.alpha_}
    
    # Use this model to predict for all days in the month
    for pred_date in month_dates:
        test_mask = model_data['date'] == pred_date
        X_test = model_data.loc[test_mask, FEATURES]
        
        if len(X_test) == 0:
            continue
        
        # Transform test features using the same scaler
        X_test_scaled = scaler.transform(X_test)
        
        test_indices = model_data.index[test_mask]
        y_pred = ridge_model.predict(X_test_scaled)
        
        for idx, pred in zip(test_indices, y_pred):
            predictions.append({
                'date': pred_date,
                'index': idx,
                'prediction': pred
            })
    
    return predictions, alpha_info

# Execute in parallel
print(f"\nRunning parallel processing with {N_JOBS} jobs...")
all_results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(train_and_predict_month)(
        month_idx, pred_month, model_data, unique_dates, oos_dates,
        FEATURES, TARGET, WINDOW, ALPHAS, CV_FOLDS
    )
    for month_idx, pred_month in enumerate(oos_months)
)

# Collect results
predictions = []
alphas = []
for month_predictions, alpha_info in all_results:
    predictions.extend(month_predictions)
    if alpha_info is not None:
        alphas.append(alpha_info)

print(f"\nCompleted.")
print(f"Total predictions: {len(predictions):,}")

Rolling window: 252 trading days
Training window: 2010-01-04 to 2011-12-31
OOS prediction period: 2012-01-03 to 2024-12-30
Number of OOS dates: 3,269
Number of OOS months: 156
Number of features: 31

Running parallel processing with 16 jobs...


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   9 tasks      | elapsed: 11.0min
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed: 18.5min
[Parallel(n_jobs=16)]: Done  29 tasks      | elapsed: 22.4min
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed: 30.9min
[Parallel(n_jobs=16)]: Done  53 tasks      | elapsed: 40.4min
[Parallel(n_jobs=16)]: Done  66 tasks      | elapsed: 49.4min
[Parallel(n_jobs=16)]: Done  81 tasks      | elapsed: 58.7min
[Parallel(n_jobs=16)]: Done  96 tasks      | elapsed: 64.3min
[Parallel(n_jobs=16)]: Done 113 tasks      | elapsed: 79.0min
[Parallel(n_jobs=16)]: Done 141 out of 156 | elapsed: 98.3min remaining: 10.5min
[Parallel(n_jobs=16)]: Done 156 out of 156 | elapsed: 104.7min finished



Completed.
Total predictions: 14,545,760


In [8]:
# Summary of selected alpha values
print("Selected Alpha (Regularization Parameter) Summary")
print("=" * 50)

if alphas:
    alphas_df = pd.DataFrame(alphas)
    print(f"\nRolling {WINDOW}-day Window:")
    print(f"  Mean alpha: {alphas_df['alpha'].mean():.6f}")
    print(f"  Std alpha: {alphas_df['alpha'].std():.6f}")
    print(f"  Min alpha: {alphas_df['alpha'].min():.6f}")
    print(f"  Max alpha: {alphas_df['alpha'].max():.6f}")

Selected Alpha (Regularization Parameter) Summary

Rolling 252-day Window:
  Mean alpha: 657075.860295
  Std alpha: 389861.514469
  Min alpha: 0.000001
  Max alpha: 1000000.000000


In [9]:
# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'prediction']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"Non-null predictions: {predictions_df['prediction'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

Predictions Summary
Total observations: 14,545,760
Non-null predictions: 14,545,760

First 10 predictions:
        date  permno ticker    index  prediction
0 2012-01-03   87432      A  2879304   -0.000142
1 2012-01-03   24643     AA  2377768   -0.000439
2 2012-01-03   12479    AAC  2266267   -0.000177
3 2012-01-03   90020   AACC  2994307   -0.000179
4 2012-01-03   15580   AAME  2343833   -0.000179
5 2012-01-03   10517    AAN  2211052   -0.000175
6 2012-01-03   76868   AAON  2576068   -0.000175
7 2012-01-03   89217    AAP  2947508   -0.000198
8 2012-01-03   14593   AAPL  2339391   -0.002933
9 2012-01-03   90854   AATI  3056541   -0.000174


In [10]:
# Save predictions to Data directory
OUTPUT_DATA_DIR = os.path.join(PROJECT_DIR, "Data")
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DATA_DIR, f"predictions_ridge_input={len(FEATURES)}.pkl")
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
if os.path.exists(OUTPUT_FILE):
    print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")

Predictions saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_ridge_input=31.pkl
File size: 535.60 MB
Shape: (14545760, 5)
Columns: ['date', 'permno', 'ticker', 'index', 'prediction']
